In [6]:
import pandas as pd
import numpy as np

# Set your grid density here. 
# 1.0 = 1 degree (~111 km distance between points)
# 0.1 = High resolution (~11 km distance)
RESOLUTION = 0.5 

print("Generating high-speed tectonic grid...")

# Comprehensive Tectonic Segments for the Indian Subcontinent & SE Asia
segments = [
    {"name": "Chile-Peru Trench", "lat_range": (-55, -5), "lon_fixed": -72},
    {"name": "Cascadia Subduction", "lat_range": (40, 50), "lon_fixed": -125},
    {"name": "Japan Trench", "lat_range": (30, 45), "lon_fixed": 142},

    {"name": "Aleutian Trench", "lat_fixed": 52, "lon_range": (-180, -130)},
    {"name": "Kuril-Kamchatka", "lat_range": (40, 55), "lon_fixed": 150},
   
    {"name": "Izu-Bonin-Mariana", "lat_range": (15, 30), "lon_fixed": 145},
    {"name": "Philippine Trench", "lat_range": (5, 20), "lon_fixed": 126},
    {"name": "Java-Sumatra Trench", "lat_range": (-10, 5), "lon_fixed": 100},
    {"name": "Tonga-Kermadec", "lat_range": (-35, -15), "lon_fixed": -175},
    {"name": "Himalayan Thrust", "lat_fixed": 28, "lon_range": (70, 100)},
    {"name": "Mediterranean-Hellenic", "lat_fixed": 35, "lon_range": (15, 30)},
    {"name": "Andaman-Nicobar Trench", "lat_range": (5, 15), "lon_fixed": 93},
    
    # --- NEW INDIAN & SE ASIAN SECTIONS ---
    # 1. The Andaman-Sumatra-Java Trench (The 2004 Tsunami Source)
    # This covers the arc from the Bay of Bengal down to Indonesia
    {"name": "Sumatra-Java Trench", "lat_range": (-10, 5), "lon_fixed": 102},
    {"name": "Andaman-Nicobar Segment", "lat_range": (6, 15), "lon_fixed": 93},
    
    # 2. Himalayan Main Frontal Thrust (Continental Collision)
    # Critical for Nepal, North India, and Bhutan
    {"name": "Indo-Gangetic Plain / Himalayan Front", "lat_range": (27,28), "lon_range": (75, 95)},
    
    # 3. Kachchh Rift & Makran Subduction (The 2001 Gujarat Trigger)
    # This covers the Gujarat coast and the active subduction off Pakistan
    {"name": "Kachchh-Gujarat Rift", "lat_fixed": 23, "lon_range": (69, 72)},
    {"name": "Makran Subduction Zone", "lat_range": (24, 25), "lon_range": (60, 70)},
    
    # 4. Indo-Burmese Arc (North East India / Myanmar)
    {"name": "Indo-Burmese Arc", "lat_range": (20, 27), "lon_fixed": 94},
    
    # 5. Central Indian Ridge (Oceanic segments influencing South India/Sri Lanka)
    {"name": "Ninety East Ridge", "lat_range": (-10, 10), "lon_fixed": 90}
]

dfs = []

# Loop only through the segment definitions (very fast)
for seg in segments:
    if "lat_range" in seg:
        # np.arange generates the entire list of coordinates instantly in C-memory
        # We add RESOLUTION/2 to the end to ensure the final coordinate is included
        lats = np.arange(seg["lat_range"][0], seg["lat_range"][1] + (RESOLUTION/2), RESOLUTION)
        lon_fixed = seg.get("lon_fixed")
        lon_range = seg.get("lon_range")

        if lon_fixed is not None:
            lon_val = lon_fixed
            # Create a vectorized DataFrame for this entire segment at once
            df_seg = pd.DataFrame({
                "zone": seg["name"],
                "latitude": np.round(lats, 4),
                "longitude": lon_val
            })
        elif lon_range is not None:
            lons = np.arange(lon_range[0], lon_range[1] + (RESOLUTION/2), RESOLUTION)
            lat_grid, lon_grid = np.meshgrid(lats, lons, indexing="ij")
            df_seg = pd.DataFrame({
                "zone": seg["name"],
                "latitude": np.round(lat_grid.ravel(), 4),
                "longitude": np.round(lon_grid.ravel(), 4)
            })
        else:
            raise ValueError(f"Segment is missing lon_fixed/lon_range: {seg['name']}")
    else:
        lons = np.arange(seg["lon_range"][0], seg["lon_range"][1] + (RESOLUTION/2), RESOLUTION)
        
        df_seg = pd.DataFrame({
            "zone": seg["name"],
            "latitude": seg["lat_fixed"],
            "longitude": np.round(lons, 4)
        })
        
    dfs.append(df_seg)

# pd.concat merges all segment blocks into one final DataFrame instantly
subduction_df = pd.concat(dfs, ignore_index=True)

# Export to CSV
file_name = 'subduction_zones.csv'
subduction_df.to_csv(file_name, index=False)

print(f"✅ Success! Generated {len(subduction_df)} grid points at {RESOLUTION}° resolution.")
print(f"File saved as '{file_name}'.")

Generating high-speed tectonic grid...
✅ Success! Generated 831 grid points at 0.5° resolution.
File saved as 'subduction_zones.csv'.


Z-Score Threshold,Rarity (Top %),Expected Occurrences in 15 Years,Real-World Meaning
Z>2.0,2.28%,≈250 Windows,An alert triggers roughly every 3 weeks.
Z>2.33 (P99),1.00%,≈109 Windows,An alert triggers roughly once every 2 months.
Z>2.5,0.62%,≈68 Windows,An alert triggers roughly 4 times a year.
Z>3.0,0.13%,≈14 Windows,An alert triggers roughly once a year.

In [ ]:


import pandas as pd
import numpy as np
import duckdb
from datetime import datetime, timedelta, timezone

# Baseline stats from your 1M sample test
SYNTHETIC_MEAN = 1.576
SYNTHETIC_STD = 0.582 

print("🚀 Initializing Precision Vectorized Forecaster (Full Physics)...")

# Load Tectonic Mask
try:
    mask = pd.read_csv('subduction_zones.csv')
    fault_lats = mask['latitude'].values
    fault_lons = mask['longitude'].values
    fault_zones = mask['zone'].values
    num_zones = len(mask)
except FileNotFoundError:
    print("Error: subduction_zones.csv not found.")
    exit()

# Time Grid: 2000 to 2035
start_date = datetime(2000, 2, 15, tzinfo=timezone.utc)
end_date = start_date + timedelta(days=365 * 35)

# Generate timestamps (every 12 hours)
time_array = np.arange(np.datetime64(start_date), np.datetime64(end_date), np.timedelta64(12, 'h'))
unix_epochs = time_array.astype('datetime64[s]').astype(np.float64)
jd_array = (unix_epochs / 86400.0) + 2440587.5
T_array = (jd_array - 2451545.0) / 36525.0

print(f"Scanning {len(time_array)} timeframes...")

# ==========================================
# RIGOROUS ASTRONOMICAL PHYSICS
# ==========================================
# Sun
L0 = 280.46646 + 36000.76983 * T_array
M_sun = 357.52911 + 35999.05029 * T_array
C_sun = 1.915 * np.sin(np.radians(M_sun))
sun_lon = (L0 + C_sun) % 360

# Accurate Moon Longitude & Latitude
L_moon = 218.316 + 481267.8813 * T_array
M_moon = 134.963 + 477198.8676 * T_array
F_moon = 93.272 + 483202.0175 * T_array # Argument of latitude
moon_lon = (L_moon + 6.289 * np.sin(np.radians(M_moon))) % 360
moon_lat = 5.128 * np.sin(np.radians(F_moon))

# Accurate Moon Declination (Accounts for the 5.14 degree orbital tilt)
epsilon = 23.439 # Earth's Obliquity
sin_dec = np.sin(np.radians(moon_lat)) * np.cos(np.radians(epsilon)) + \
          np.cos(np.radians(moon_lat)) * np.sin(np.radians(epsilon)) * np.sin(np.radians(moon_lon))
moon_dec_accurate = np.degrees(np.arcsin(sin_dec))

# Planets
def get_planet_lon(L_m, pi, e):
    return (L_m + np.degrees(2 * e * np.sin(np.radians(L_m - pi)))) % 360

mars_lon = get_planet_lon(355.43 + 19140.3 * T_array, 336.06, 0.0934)
jup_lon = get_planet_lon(34.35 + 3034.9 * T_array, 14.33, 0.0485)
sat_lon = get_planet_lon(50.08 + 1222.1 * T_array, 92.06, 0.0555)

# ==========================================
# SCORE CALCULATION
# ==========================================
syzygy = (1 + np.cos(np.radians(2 * np.abs(moon_lon - sun_lon)))) / 2

# S-Score Array
s_scores = (0.13 * syzygy) + (0.10 * np.abs(moon_dec_accurate)) - \
           (0.28 * np.sin(np.radians(sat_lon))) - \
           (0.15 * np.sin(np.radians(jup_lon))) + \
           (0.04 * np.sin(np.radians(mars_lon)))

# Z-Score Array
z_scores = (s_scores - SYNTHETIC_MEAN) / SYNTHETIC_STD

# ==========================================
# FILTERING & PERSISTENCE
# ==========================================
# Find ONLY the indices where Z-Score > 0.1
critical_indices = np.where(z_scores > 0.1)[0]

##print(f"🎯 Identified {len(critical_indices)} critical temporal windows. Applying spatial mapping...")

forecast_results = []

for idx in critical_indices:
    critical_time = pd.Timestamp(time_array[idx])
    critical_z = z_scores[idx]
    
    risk_level = 'HIGH' if critical_z > 2.33 else 'ELEVATED'
    
    batch = pd.DataFrame({
        'timestamp': [critical_time] * num_zones,
        'zone': fault_zones,
        'lat': fault_lats,
        'lon': fault_lons,
        'z_score': [round(critical_z, 2)] * num_zones,
        'risk_level': [risk_level] * num_zones
    })
    forecast_results.append(batch)

if forecast_results:
    df_forecast = pd.concat(forecast_results, ignore_index=True)
    
    con = duckdb.connect('mega_quake_unified.db')
    con.execute("CREATE OR REPLACE TABLE forecast_details AS SELECT * FROM df_forecast")
    
    ##extreme_df = df_forecast[df_forecast['z_score'] > 3.0]
    extreme_df.to_csv('Top_Risk_Windows_2026_2041.csv', index=False)
    
    print(f"\n✅ Forecast Complete. Identified {len(critical_indices)} elevated windows.")
    print(f"✅ Found {len(extreme_df['timestamp'].unique())} EXTREME ($Z > 2.5$) risk dates.")
    print("Results saved instantly to 'mega_quake_unified.db' and CSV.")
else:
    print("\n✅ Forecast Complete. No high-risk windows identified.")

con.close()

🚀 Initializing Precision Vectorized Forecaster (Full Physics)...
Scanning 25550 timeframes...
🎯 Identified 1395 critical temporal windows. Applying spatial mapping...


/tmp/ipykernel_2337/2613969171.py:28: UserWarning: no explicit representation of timezones available for np.datetime64
  time_array = np.arange(np.datetime64(start_date), np.datetime64(end_date), np.timedelta64(12, 'h'))



✅ Forecast Complete. Identified 1395 elevated windows.
✅ Found 0 EXTREME ($Z > 2.5$) risk dates.
Results saved instantly to 'mega_quake_unified.db' and CSV.


In [19]:
con = duckdb.connect('mega_quake_unified.db')
res_df = con.execute("SELECT * FROM forecast_details").df()
print(res_df.head())


   timestamp               zone   lat   lon  z_score risk_level
0 2005-06-22  Chile-Peru Trench -55.0 -72.0      2.0   ELEVATED
1 2005-06-22  Chile-Peru Trench -54.5 -72.0      2.0   ELEVATED
2 2005-06-22  Chile-Peru Trench -54.0 -72.0      2.0   ELEVATED
3 2005-06-22  Chile-Peru Trench -53.5 -72.0      2.0   ELEVATED
4 2005-06-22  Chile-Peru Trench -53.0 -72.0      2.0   ELEVATED
